# 07 · Exploratory PCA of mutational profiles

Looks for recurring amino-acid preferences across the positions of each LxCxE peptide, beyond the
canonical motif, in the spirit of Tsuboyama et al. (2023) (report, section 5).

For every peptide a matrix is built with **one row per mutated position** (14 rows) and
**one column per substituting amino acid** (20 columns). Each cell is the experimental ΔZ-score of that
substitution (the WT residue itself is filled with 0). A PCA (scikit-learn) is fitted on the
standardised columns and components are retained according to the explained variance.

Two complementary views are produced:

1. **Position × amino acid matrix** → which positions behave alike; the loadings show which amino acids
   drive each component. A variant excluding the core region (positions 5–9) is used for the
   multilayer heatmaps and loadings.
2. **Transposed matrix** (amino acid × position) → which amino acids behave alike across positions.

**Input:** `data/processed/df_dms.csv` (notebook 01).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Project paths
#   * Locally: start Jupyter from the `notebooks/` folder.
#   * Google Colab: Google Drive is mounted; edit the folder name below so it
#     points to the project folder stored in your Drive.
#   * Anywhere: set the PROJECT_DIR environment variable to override.
# ---------------------------------------------------------------------------
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/plm-embeddings-lxcxe-prb")
except ImportError:
    DEFAULT_PROJECT_DIR = Path.cwd().parent

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", DEFAULT_PROJECT_DIR))
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"
for folder in (PROCESSED_DIR, RESULTS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def save_figure(name):
    """Save the current matplotlib figure into figures/ (call before plt.show())."""
    plt.savefig(FIGURES_DIR / name, dpi=300, bbox_inches="tight")

import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

## 1. Load data and build the per-protein matrices

In [ ]:
df_dms = pd.read_csv(PROCESSED_DIR / "df_dms.csv")
PROTEINS = df_dms["protein"].drop_duplicates().tolist()

AA_ORDER = list("HKRDENQSTCMAILVFWYGP")
AA_GROUP_BOUNDARIES = [4, 8, 11, 16, 19]  # horizontal separators in the DMS heatmaps
CORE_REGION = range(5, 10)  # positions 5-9 (LxCxE core) removed in the "core excluded" analysis
VARIANCE_THRESHOLD = 0.85

In [ ]:
def build_dms_matrices(df):
    """{protein: DataFrame(position label x mutant amino acid)} with ΔZ values (pm_rb)."""
    matrices = {}
    for protein, sub in df.groupby("protein", sort=False):
        sub = sub.assign(label=sub["wt_aa"] + sub["position"].astype(str))
        order = sub.sort_values("position")["label"].unique()
        matrix = sub.pivot(index="label", columns="mut_aa", values="delta_pm_rb")
        matrices[protein] = matrix.reindex(index=order, columns=AA_ORDER).fillna(0)
    return matrices


dms_matrices = build_dms_matrices(df_dms)
dms_matrices[PROTEINS[0]].head()

## 2. PCA functions

`matrix` is always *observations × features*. For the main analysis observations are positions and
features are amino acids; for the transposed analysis the roles are swapped.

In [ ]:
def drop_core(matrix):
    """Remove the rows of the LxCxE core region (positions 5-9)."""
    positions = matrix.index.str.extract(r"(\d+)")[0].astype(int)
    return matrix[~positions.isin(CORE_REGION).values]


def explained_variance_table(matrix, threshold=VARIANCE_THRESHOLD):
    """Fit a PCA on standardised columns; return variances and #PCs reaching `threshold`."""
    scaled = StandardScaler().fit_transform(matrix.values)
    pca = PCA(random_state=42).fit(scaled)
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    return {
        "explained": pca.explained_variance_ratio_,
        "cumulative": cumulative,
        "n_retained": int(np.argmax(cumulative >= threshold) + 1),
    }


def fit_pca(matrix, n_components):
    """PCA with `n_components` on the standardised matrix; keeps what the plots need."""
    scaled = StandardScaler().fit_transform(matrix.values)
    pca = PCA(n_components=n_components, random_state=42).fit(scaled)
    scores = pd.DataFrame(pca.transform(scaled), columns=[f"PC{k + 1}" for k in range(n_components)])
    scores["label"] = matrix.index.values
    return {
        "pca": pca,
        "scores": scores,
        "features": list(matrix.columns),
        "values": matrix.values.T,  # features x observations (for the heatmap)
    }

In [ ]:
def plot_scree(protein, variance, title_prefix="PCA scree plot", save_as=None):
    components = np.arange(1, len(variance["explained"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(components, variance["explained"], "o-")
    plt.axvline(variance["n_retained"], linestyle="--", color="gray",
                label=f"{VARIANCE_THRESHOLD:.0%} variance ({variance['n_retained']} PCs)")
    plt.xticks(components)
    plt.xlabel("Principal component")
    plt.ylabel("Explained variance ratio")
    plt.title(f"{title_prefix} — {protein}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    if save_as:
        save_figure(save_as)
    plt.show()


def plot_pc_heatmaps(protein, result, save=True):
    """DMS heatmap (amino acid x position) ordered by each PC, with the PC score track on top."""
    scores, values, features = result["scores"], result["values"], result["features"]
    for pc in [c for c in scores.columns if c.startswith("PC")]:
        order = np.argsort(scores[pc].values)

        fig = plt.figure(figsize=(16, 9), dpi=150)
        grid = gridspec.GridSpec(2, 2, height_ratios=[0.5, 4], width_ratios=[32, 1],
                                 wspace=0.02, hspace=0.06)
        ax_pc, ax_dms = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[1, 0])
        cax_pc, cax_dms = fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, 1])

        sns.heatmap(scores[[pc]].values[order].T, ax=ax_pc, cbar_ax=cax_pc, cmap="coolwarm_r",
                    xticklabels=False, yticklabels=[pc], cbar_kws={"label": "PC score"})
        for tick in ax_pc.get_yticklabels():
            tick.set_color("red")
            tick.set_weight("bold")
        ax_pc.set_title(f"DMS profile ({protein}) — ordered by {pc}", fontsize=13, pad=12, weight="bold")

        sns.heatmap(values[:, order], ax=ax_dms, cbar_ax=cax_dms, cmap="bwr", center=0,
                    xticklabels=scores["label"].values[order], yticklabels=features,
                    cbar_kws={"label": "ΔZ-score"})
        ax_dms.tick_params(axis="x", rotation=90, labelsize=7)
        ax_dms.tick_params(axis="y", rotation=0, labelsize=9)
        ax_dms.set_xlabel(f"Positions, ordered by {pc} score", fontsize=10, labelpad=8)
        ax_dms.set_ylabel("Mutant amino acid", fontsize=10)
        for boundary in AA_GROUP_BOUNDARIES:
            ax_dms.axhline(y=boundary, color="black", linewidth=1.2)
        if save:
            save_figure(f"pca_heatmap_{protein}_{pc}.png")
        plt.show()


def plot_loadings(protein, result, feature_label="amino acid", save_as=None):
    """Bar plot of the loadings of each retained PC, sorted from low to high."""
    pca, features = result["pca"], np.array(result["features"])
    n_pcs = pca.n_components_
    palette = ["#72a3d0", "#f6a96e", "#72be85", "#e37778", "#b08ecb", "#a67c1e", "#e377c2"]

    fig, axes = plt.subplots(1, n_pcs, figsize=(3.8 * n_pcs, 2.8), sharey=True, dpi=150)
    axes = np.atleast_1d(axes)
    for i, ax in enumerate(axes):
        order = np.argsort(pca.components_[i])
        loadings, labels = pca.components_[i][order], features[order]
        x = np.arange(len(labels))
        ax.bar(x, loadings, color=palette[i % len(palette)], width=0.85)
        ax.axhline(0, color="black", linewidth=1.2)
        for xi, value, label in zip(x, loadings, labels):
            ax.text(xi, value + (0.015 if value >= 0 else -0.015), label, ha="center",
                    va="bottom" if value >= 0 else "top", fontsize=8, fontweight="bold",
                    color="#333333")
        ax.set_title(f"{protein}\nPC{i + 1}: {pca.explained_variance_ratio_[i]:.0%}",
                     fontsize=11, fontweight="bold", pad=10)
        ax.set_xticks([])
        for side in ("top", "right", "bottom"):
            ax.spines[side].set_visible(False)
        ax.spines["left"].set_visible(i == 0)
        ax.set_ylim(min(loadings.min(), 0) - 0.08, max(loadings.max(), 0) + 0.08)
    axes[0].set_ylabel(f"Loading ({feature_label})", fontsize=9)
    plt.tight_layout()
    if save_as:
        save_figure(save_as)
    plt.show()

## 3. How many components? (all positions)

Scree plots; the dashed line marks the number of PCs needed to reach 85 % of the variance.

In [ ]:
for protein in PROTEINS:
    variance = explained_variance_table(dms_matrices[protein])
    print(f"{protein}: {variance['n_retained']} PCs retained, "
          f"explained variance = {np.round(variance['explained'][:variance['n_retained']], 3)}")
    plot_scree(protein, variance, save_as=f"pca_scree_{protein}.png")

## 4. Positions × amino acids, core region excluded

Excluding positions 5–9 highlights preferences in the flanking residues. The first three PCs are shown:
the heatmaps project each component along the peptide, and the loadings reveal which amino acids define it.

In [ ]:
for protein in PROTEINS:
    result = fit_pca(drop_core(dms_matrices[protein]), n_components=3)
    plot_pc_heatmaps(protein, result)
    plot_loadings(protein, result, save_as=f"pca_loadings_{protein}.png")

## 5. Transposed matrix (amino acids × positions)

Here each observation is a substituting amino acid and the features are the positions, so the loadings
show which positions share the same amino-acid preferences.

In [ ]:
for protein in PROTEINS:
    transposed = dms_matrices[protein].T
    variance = explained_variance_table(transposed)
    plot_scree(protein, variance, title_prefix="PCA scree plot (transposed)",
               save_as=f"pca_scree_transposed_{protein}.png")

    result = fit_pca(transposed, n_components=4)
    plot_loadings(protein, result, feature_label="position",
                  save_as=f"pca_loadings_transposed_{protein}.png")